# 02 Data Preprocessing

Validate schemas, fit preprocessors on the training split only, and inspect missing-value handling.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
TRAINING = ROOT / 'training'
DATA = ROOT / 'datasets'
sys.path.insert(0, str(TRAINING))

from preprocessing import TrafficPreprocessor, TelemetryPreprocessor, IncidentPreprocessor, canonical_link_id_map, load_capacities, load_physical_baselines, load_universe_config
from split import tick_split

In [ ]:
cfg = load_universe_config(DATA / 'universe-config.json')
baselines = load_physical_baselines(cfg)
capacities = load_capacities(cfg)
link_id_map = canonical_link_id_map(cfg)

traffic = pd.read_csv(DATA / 'link_traffic_history.csv').sort_values(['link_id', 'tick'])
telemetry = pd.read_csv(DATA / 'link_telemetry.csv').sort_values(['link_id', 'tick'])
incidents = pd.read_csv(DATA / 'link_incident_history.csv').sort_values(['link_id', 'tick'])

In [ ]:
traffic_split = tick_split(traffic)
telemetry_split = tick_split(telemetry)
incident_split = tick_split(incidents)

traffic_prep = TrafficPreprocessor().fit(traffic_split.train, baselines, capacities, link_id_map)
telemetry_prep = TelemetryPreprocessor().fit(telemetry_split.train, baselines, link_id_map)
incident_prep = IncidentPreprocessor().fit(incident_split.train, link_id_map)

display(traffic_prep.metadata())
display(telemetry_prep.metadata())
display(incident_prep.metadata())

In [ ]:
traffic_processed = traffic_prep.transform(traffic)
telemetry_processed = telemetry_prep.transform(telemetry)
incident_processed = incident_prep.transform(incidents)

display(traffic_processed[['load_units_missing', 'observed_latency_missing', 'is_saturated']].sum())
display(telemetry_processed[['self_reported_latency_missing']].sum())
display(incident_processed[['traffic_share_missing']].sum())